In [ ]:
"""
PROJECT: Can a computer predict if a molecule dissolves in water?

WHAT THIS SCRIPT DOES (in 6 steps)
  1. Load a public list of molecules with measured solubility.
  2. Turn each molecule into numbers (computers only understand numbers).
  3. Split the data into a "learning" part and a "test" part - in TWO different ways.
  4. Train three models on the learning part.
  5. Check how wrong each model is on the test part.
  6. Save a chart and a results table.

To run:  python solubility_simple.py
"""

import urllib.request
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

URL = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
FILE = Path("delaney-processed.csv")
TEST_FRACTION = 0.2      # 20% of molecules are kept back for testing
SEEDS = [0, 1, 2]        # repeat 3 times with different shuffles so one lucky split does not fool us


# ---------------------------------------------------------------- STEP 1: load data
def load_data():
    if not FILE.exists():
        print("Downloading the dataset...")
        urllib.request.urlretrieve(URL, FILE)
    df = pd.read_csv(FILE)
    # keep only the two columns we need: the molecule (SMILES text) and its measured solubility
    df = df[["smiles", "measured log solubility in mols per litre"]]
    df.columns = ["smiles", "solubility"]
    return df


# ---------------------------------------------------------------- STEP 2: molecules -> numbers
def molecule_to_numbers(smiles):
    """SMILES is a text code for a molecule, e.g. 'CCO' is ethanol."""
    mol = Chem.MolFromSmiles(smiles)

    # A fingerprint is a long list of 0s and 1s saying which small pieces the molecule contains.
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
    fingerprint = np.zeros(1024)
    DataStructs.ConvertToNumpyArray(generator.GetFingerprint(mol), fingerprint)

    # Descriptors are a few well-known properties of the molecule (size, greasiness, ...).
    descriptors = [
        Descriptors.MolLogP(mol),    # how oily/greasy it is (oily molecules dissolve badly in water)
        Descriptors.MolWt(mol),      # how heavy it is
        Descriptors.TPSA(mol),       # how much of its surface is water-loving
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
    ]

    # Scaffold = the ring-shaped "skeleton" of the molecule. Used to group chemical families.
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    return fingerprint, np.array(descriptors), scaffold


# ---------------------------------------------------------------- STEP 3: two ways to split
def random_split(n, seed):
    """EASY TEST: pick test molecules completely at random."""
    order = np.random.RandomState(seed).permutation(n)
    n_test = int(n * TEST_FRACTION)
    return order[n_test:], order[:n_test]          # (learning rows, test rows)


def scaffold_split(scaffolds, seed):
    """HARD TEST: whole chemical families go to the test set, so the test molecules are
    structurally NEW to the model - like real drug discovery."""
    families = defaultdict(list)
    for row, scaffold in enumerate(scaffolds):
        families[scaffold].append(row)

    names = list(families)
    np.random.RandomState(seed).shuffle(names)

    max_test = int(len(scaffolds) * TEST_FRACTION)
    test = []
    for name in names:
        if len(test) + len(families[name]) <= max_test:
            test.extend(families[name])

    learning = [row for row in range(len(scaffolds)) if row not in set(test)]
    return np.array(learning), np.array(test)


# ---------------------------------------------------------------- STEPS 4 + 5: train and check
def rmse(true, predicted):
    """RMSE = typical size of the mistake, in the same units as solubility. Lower is better."""
    return float(np.sqrt(mean_squared_error(true, predicted)))


def evaluate(split_function, split_name, X_fp, X_desc, y, seed, results):
    learn, test = split_function(seed)

    # Model 1: ALWAYS guess the average solubility. If a real model cannot beat this, it has learned nothing.
    baseline = DummyRegressor(strategy="mean").fit(X_desc[learn], y[learn])
    results.append((split_name, seed, "1. Always guess the average", rmse(y[test], baseline.predict(X_desc[test]))))

    # Model 2: Ridge - a simple straight-line style model using the property numbers.
    scaler = StandardScaler().fit(X_desc[learn])               # learn the scaling from learning rows only
    ridge = Ridge().fit(scaler.transform(X_desc[learn]), y[learn])
    results.append((split_name, seed, "2. Ridge (properties)", rmse(y[test], ridge.predict(scaler.transform(X_desc[test])))))

    # Model 3: Random forest - many decision trees voting, using the fingerprints.
    forest = RandomForestRegressor(n_estimators=100, random_state=seed).fit(X_fp[learn], y[learn])
    results.append((split_name, seed, "3. Random forest (fingerprints)", rmse(y[test], forest.predict(X_fp[test]))))


def main():
    df = load_data()

    # STEP 2: convert every molecule
    converted = [molecule_to_numbers(s) for s in df["smiles"]]
    X_fp = np.array([c[0] for c in converted])
    X_desc = np.array([c[1] for c in converted])
    scaffolds = [c[2] for c in converted]
    y = df["solubility"].to_numpy()
    print(f"Loaded {len(y)} molecules.")

    # STEPS 3-5: run both tests, several times each
    results = []
    for seed in SEEDS:
        evaluate(lambda s: random_split(len(y), s), "Easy test (random)", X_fp, X_desc, y, seed, results)
        evaluate(lambda s: scaffold_split(scaffolds, s), "Hard test (new families)", X_fp, X_desc, y, seed, results)

    table = pd.DataFrame(results, columns=["test", "seed", "model", "error_rmse"])
    summary = table.groupby(["model", "test"])["error_rmse"].mean().unstack().round(3)
    print("\nAverage error (lower is better):\n")
    print(summary)
    summary.to_csv("results_table.csv")

    # STEP 6: chart
    summary.plot(kind="bar", figsize=(9, 5))
    plt.ylabel("Average error (lower is better)")
    plt.title("A simple model holds up on new molecules, but the random forest falls behind")
    plt.xticks(rotation=0, ha="center")
    plt.tight_layout()
    plt.savefig("results_chart.png", dpi=150)
    print("\nSaved results_table.csv and results_chart.png")


if __name__ == "__main__":
    main()

Loaded 1128 molecules.
